# 🏔️ OE Data Intelligence — Medallion Pipeline

**Full end-to-end pipeline: raw sys logs → Bronze → Silver → Gold KPI tables**

## Prerequisites
1. All Docker containers running: `docker compose up -d`
2. Mock logs generated (see Step 1)

## Data Sources
| Source | Log Format | Issues Simulated |
|---|---|---|
| **Aternity** | JSONL | Teams/Outlook degradation, high response times |
| **NetScout** | CSV | WAN packet loss, congestion, hardware faults |
| **Intune** | JSONL | Non-compliant devices, Win7/8, MDM check-in failures |
| **Infoblox** | Syslog (RFC 5424) | DNS latency, NXDOMAIN, suspicious C2 queries |
| **Salesforce** | CSV | P1/P2 incidents, SLA breaches, recurring issues |
| **ScienceLogic** | JSONL | Infrastructure alerts, threshold breaches |

## ⚠️ Important: Spark Local Mode
The pipeline runs in `local[2]` mode (2 threads inside Jupyter JVM).
This avoids a Java serialization error caused by version mismatch:
- `jupyter/pyspark-notebook` → Spark **3.5.0**
- `apache/spark` worker → Spark **3.5.3**

On real Databricks (AWS GovCloud), this is not an issue — single runtime version.

## Step 1 — Generate Mock Log Files

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '/home/jovyan/generate_mock_logs.py', '--days', '1'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 2 — Preview Raw Logs

In [ ]:
import json
from pathlib import Path

print('=== Aternity (JSONL — first 3 records) ===')
logs = list(Path('/home/jovyan/mock-logs/aternity').glob('*.jsonl'))
if logs:
    with open(logs[0]) as fp:
        for i, line in enumerate(fp):
            if i < 3:
                print(json.dumps(json.loads(line), indent=2))
else:
    print('No Aternity logs found — run Step 1 first')

In [ ]:
print('=== Infoblox DNS Syslog (RFC 5424 — first 5 lines) ===')
logs = list(Path('/home/jovyan/mock-logs/infoblox').glob('*.log'))
if logs:
    with open(logs[0]) as f:
        for i, line in enumerate(f):
            if i < 5:
                print(line.strip())
else:
    print('No Infoblox logs found — run Step 1 first')

## Step 3 — Initialize Spark + Run Full Pipeline

**Note:** `os.environ["SPARK_MASTER"] = "local[2]"` must be set BEFORE
importing spark_session. This overrides the container env var
`SPARK_MASTER=spark://spark-master:7077` which causes a version mismatch error.

In [ ]:
import os
# CRITICAL: set before any spark import to force local mode
os.environ["SPARK_MASTER"] = "local[2]"

import sys
sys.path.insert(0, '/home/jovyan')

from spark_session import get_spark, Paths

spark = get_spark('OE-Pipeline-Notebook')
print('Spark ready:', spark.version)
print('Master:',      spark.sparkContext.master)
# Must show: Master: local[2]

In [ ]:
from pipelines.run_pipeline import run
run()

## Step 4 — Query Gold Tables (what the Spring Boot API serves)

**Note:** The pipeline calls `spark.stop()` at the end. This cell
reconnects a fresh SparkSession before reading the Gold tables.

In [ ]:
import os
os.environ["SPARK_MASTER"] = "local[2]"

import sys
sys.path.insert(0, '/home/jovyan')

from spark_session import get_spark, Paths
spark = get_spark('OE-Query')
print('Spark ready for queries:', spark.version)

In [ ]:
print('=== App Health Summary (worst first) ===')
spark.read.format('delta').load(Paths.gold('app_health_summary')) \
    .select('app_name','health_status','experience_score',
            'avg_response_time_ms','crash_rate','active_user_count') \
    .orderBy('experience_score') \
    .show(10, truncate=False)

In [ ]:
print('=== Packet Loss Root Cause Analysis ===')
spark.read.format('delta').load(Paths.gold('packet_loss_root_cause')) \
    .select('segment_name','root_cause','packet_loss_rate',
            'confidence_score','severity','recommendation') \
    .orderBy('packet_loss_rate', ascending=False) \
    .show(truncate=False)

In [ ]:
print('=== Non-Compliant Devices ===')
spark.read.format('delta').load(Paths.gold('device_health_summary')) \
    .filter('compliance_state = "NON_COMPLIANT"') \
    .select('device_name','os_name','os_version','compliance_state',
            'days_since_checkin','assigned_user','location') \
    .orderBy('days_since_checkin', ascending=False) \
    .show(10, truncate=False)

In [ ]:
print('=== DNS Server Metrics (Infoblox) ===')
spark.read.format('delta').load(Paths.gold('dns_metrics')) \
    .select('server','total_queries','failed_queries',
            'failure_rate','avg_response_ms','nxdomain_count') \
    .orderBy('avg_response_ms', ascending=False) \
    .show(truncate=False)

In [ ]:
print('=== Critical Open Issues (P1/P2) ===')
spark.read.format('delta').load(Paths.gold('top_issues_summary')) \
    .filter('severity IN ("P1","P2") AND status != "RESOLVED"') \
    .select('issue_id','severity','title','category','assigned_team') \
    .orderBy('severity') \
    .show(truncate=False)

In [ ]:
print('=== Data Source Ingestion Status ===')
spark.read.format('delta').load(Paths.gold('data_source_ingestion_status')) \
    .select('source_name','status','records_last_batch',
            'data_quality_score','latency_ms','error_message') \
    .show(truncate=False)

In [ ]:
print('=== Version Sprawl — OS Distribution ===')
spark.read.format('delta').load(Paths.gold('version_sprawl_summary')) \
    .select('software_name','version','device_count',
            'is_supported','update_urgency') \
    .orderBy('device_count', ascending=False) \
    .show(truncate=False)